# Trabalho Parcial - Ciência de Dados e Inteligência Artificial

Gabriel Ceni
Programação Avançada - 2026.2

## 1. Instalação e Importação de Bibliotecas

In [91]:
# Instalação pacotes necessários em modo silencioso (-q)
!pip install ucimlrepo plotly scikit-learn -q

In [92]:
#
import pandas as pd
import pandas.io.formats.style
import numpy as np
import hashlib

from typing import Optional, Literal

# Dados
from ucimlrepo import fetch_ucirepo

### Import Plots

In [93]:
# Visualização
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go

from plotly.subplots import make_subplots

### Import MLs - SKLearn

In [94]:
# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler, OneHotEncoder

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.decomposition import PCA

import joblib

In [95]:
print(pd.__version__)

2.2.3


### Definições Globais

In [96]:
# Ajuste do Plotly para Colab
pio.renderers.default = 'colab'

# Definição de Cores/Temas
WINE_COLORS = {
    'red': '#722F37',   # Wine Plum
    'white': '#EDE8D0'  # Vanilla Crea
}

SPOILED_COLORS = {
    'red': '#5C3B39',   # Chocolate Plum
    'white': '#895129'  # Saddle Brown
}

In [97]:
def style_wine_df(
    df: pd.DataFrame,
    n_rows: Optional[int] = None,
    method: Literal["head", "sample"] = "head",
    quality_threshold: int = 6,
    random_state: Optional[int] = 33
) -> pandas.io.formats.style.Styler:
    """
    Estiliza um DataFrame de vinhos colorindo as linhas de acordo com a cor e a qualidade.

    - Usa WINE_COLORS quando quality >= quality_threshold
    - Usa SPOILED_COLORS quando quality < quality_threshold

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame contendo as colunas 'color' e 'quality'.
    n_rows : int, optional
        Quantidade de linhas a exibir. Se None, exibe todas.
    method : {"head", "sample"}, default="head"
        Método de seleção das linhas:
        - "head": pega as primeiras n_rows
        - "sample": pega n_rows aleatórias
    quality_threshold : int, default=6
        Limite de qualidade para decidir entre cores normais e "spoiled".
    random_state : int, optional
        Semente para o sample (garante reprodutibilidade). Só usado quando method="sample".

    Returns
    -------
    pd.io.formats.style.Styler
        Objeto Styler pronto para ser exibido com display().
    """
    if n_rows is None:
        data = df
    elif method == "sample":
        data = df.sample(n=n_rows, random_state=random_state)
    else:  # method == "head"
        data = df.head(n_rows)

    def apply_color(row: pd.Series) -> list[str]:
        is_spoiled = row['quality'] < quality_threshold
        color_map = SPOILED_COLORS if is_spoiled else WINE_COLORS

        bg_color = color_map.get(row['color'], '#FFFFFF')
        text_color = 'white' if row['color'] == 'red' else 'black'

        return [f'background-color: {bg_color}; color: {text_color}'] * len(row)

    return (
        data.style
        .apply(apply_color, axis=1)
        .set_properties(**{'text-align': 'center'})
        .format(precision=2)
        .set_caption("Wine Dataset — Colorido por tipo e qualidade")
    )

## 2. Visualização Prévia e Tratamento de Dados

In [98]:
# Carregar o dataset Wine Quality
wine_quality = fetch_ucirepo(id=186)

In [99]:
wine_df = wine_quality.data.original
# display(wine_df.head(3))
display(style_wine_df(wine_df, n_rows=3, method="head"))
print("=== Amostra ===")
# display(wine_df.head(20))
display(style_wine_df(wine_df, n_rows=20, method="sample"))
print("=== Describe ===")
display(wine_df.describe())

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality,color
0,7.40,0.70,0.00,1.90,0.08,11.00,34.00,1.00,3.51,0.56,9.40,5,red
1,7.80,0.88,0.00,2.60,0.10,25.00,67.00,1.00,3.20,0.68,9.80,5,red
2,7.80,0.76,0.04,2.30,0.09,15.00,54.00,1.00,3.26,0.65,9.80,5,red


=== Amostra ===


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality,color
5530,6.20,0.30,0.28,1.60,0.04,28.00,106.00,0.99,3.14,0.41,13.30,6,white
3603,7.40,0.26,0.31,2.40,0.04,58.00,178.00,0.99,3.42,0.68,10.60,6,white
4126,7.30,0.16,0.35,1.50,0.04,29.00,108.00,0.99,3.27,0.51,10.20,6,white
2582,5.70,0.36,0.34,4.20,0.03,21.00,77.00,0.99,3.41,0.45,11.90,6,white
5451,5.90,0.20,0.23,1.50,0.04,38.00,93.00,0.99,3.36,0.49,12.00,6,white
5223,6.60,0.36,0.28,6.10,0.03,12.00,93.00,0.99,3.19,0.27,12.80,7,white
3938,7.00,0.23,0.26,7.20,0.04,21.00,90.00,1.00,3.22,0.55,9.50,6,white
1363,8.00,0.83,0.27,2.00,0.08,11.00,63.00,1.00,3.29,0.48,9.80,4,red
5728,6.30,0.34,0.31,6.00,0.02,18.00,68.00,0.99,3.22,0.29,13.40,7,white
757,8.10,0.87,0.00,2.20,0.08,10.00,31.00,1.00,3.25,0.50,9.80,5,red


=== Describe ===


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality
count,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000
mean,7.215307,0.339666,0.318633,5.443235,0.056034,30.525319,115.744574,0.994697,3.218501,0.531268,10.491801,5.818378
std,1.296434,0.164636,0.145318,4.757804,0.035034,17.749400,56.521855,0.002999,0.160787,0.148806,1.192712,0.873255
min,3.800000,0.080000,0.000000,0.600000,0.009000,1.000000,6.000000,0.987110,2.720000,0.220000,8.000000,3.000000
25%,6.400000,0.230000,0.250000,1.800000,0.038000,17.000000,77.000000,0.992340,3.110000,0.430000,9.500000,5.000000
50%,7.000000,0.290000,0.310000,3.000000,0.047000,29.000000,118.000000,0.994890,3.210000,0.510000,10.300000,6.000000
75%,7.700000,0.400000,0.390000,8.100000,0.065000,41.000000,156.000000,0.996990,3.320000,0.600000,11.300000,6.000000
max,15.900000,1.580000,1.660000,65.800000,0.611000,289.000000,440.000000,1.038980,4.010000,2.000000,14.900000,9.000000


### 2.1 Tratamento de Dados

Verificação de valores nulos e duplicados.

In [100]:
# Verificação de valores nulos
print("Linhas Nulas?\n", wine_df.isnull().sum())
print("Linhas Nulas?\n", wine_df.isnull().sum().sum(), " / ", wine_df.shape[0])

Linhas Nulas?
 fixed_acidity           0
volatile_acidity        0
citric_acid             0
residual_sugar          0
chlorides               0
free_sulfur_dioxide     0
total_sulfur_dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
color                   0
dtype: int64
Linhas Nulas?
 0  /  6497


In [101]:
# Verificação e remoção de dados duplicados

print("Valores Duplicados?\n", wine_df.duplicated().sum(), " / ", wine_df.shape[0])
wine_uniq_df = wine_df.drop_duplicates()
print("DataFrame Sem Duplicados?\n", wine_uniq_df.duplicated().sum(), " / ", wine_uniq_df.shape[0])

Valores Duplicados?
 1177  /  6497
DataFrame Sem Duplicados?
 0  /  5320


A presença de bastante linhas duplicadas leva a decidir entre:

- Remover linhas duplicadas:
    - **Prevenção de Vazamento de Dados** (*Data Leakage*) - Evitar cópias exatas do mesmo vinho.
    - **Redução de Viés** - Impedir que o algoritmo dê um peso desproporcional a um perfil de vinho específico.
    - **Segurança da Avaliação** - Garantir que o modelo seja testado de forma honesta contra dados que ele realmente nunca viu.
- Manter linhas duplicadas:
    - **Possibilidade de Coincidência Legítima** - Ausência de identificadores não permitem obter certeza se são os mesmos vinhos ou apenas similares.
    - **Preservação da Distribuição Natural** - Se as duplicatas representam a realidade, então equivalem à frequência real que a IA deva aprender.

### 2.2 Atributos (features)

- **fixed_acidity:** Ácidos que não evaporam facilmente. São fundamentais para garantir a acidez base e a estrutura geral do vinho.

- **volatile_acidity:** Quantidade de ácido acético. Em níveis elevados, confere ao vinho um cheiro e sabor indesejáveis de vinagre.

- **citric_acid:** Presente em pequenas quantidades, o ácido cítrico adiciona frescor e notas de sabor ao vinho.

- **residual_sugar:** O açúcar que permanece no vinho após o fim da fermentação. Define o nível de doçura da bebida.

- **chlorides:** A quantidade de sal (cloretos) dissolvida no vinho.

- **free_sulfur_dioxide:** Dióxido de enxofre em estado livre. Atua como agente antimicrobiano e evita a oxidação precoce do vinho.

- **total_sulfur_dioxide:** A soma do dióxido de enxofre livre com o ligado a outras moléculas. Em concentrações muito altas, afeta negativamente o cheiro e o sabor.

- **density:** A densidade do líquido. Varia diretamente de acordo com a quantidade de açúcar e inversamente em relação ao teor de álcool.

- **pH:** Mede quão ácido ou básico o vinho é. A escala vai de 0 a 14, mas a grande maioria dos vinhos situa-se entre 3.0 e 4.0.

- **sulphates:** Aditivos que contribuem para a manutenção dos níveis de dióxido de enxofre, funcionando como conservantes antioxidantes.

- **alcohol:** O percentual de álcool por volume presente na bebida.

In [102]:
# features
X_df = wine_uniq_df.drop(columns=['quality', 'color'])
print("=== Head ===")
display(X_df.sample(10))
print("=== Describe ===")
display(X_df.describe())
print("=== Info ===")
print(X_df.info())


=== Head ===


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol
1747,6.3,0.340,0.33,4.60,0.034,19.0,80.0,0.99170,3.38,0.58,12.00
268,6.9,0.540,0.04,3.00,0.077,7.0,27.0,0.99870,3.69,0.91,9.40
537,8.1,0.825,0.24,2.10,0.084,5.0,13.0,0.99720,3.37,0.77,10.70
6305,6.1,0.220,0.40,1.85,0.031,25.0,111.0,0.98966,3.03,0.30,11.80
6082,5.8,0.280,0.18,1.20,0.058,7.0,108.0,0.99288,3.23,0.58,9.55
662,7.2,0.570,0.06,1.60,0.076,9.0,27.0,0.99720,3.36,0.70,9.60
4579,6.1,0.380,0.42,5.00,0.016,31.0,113.0,0.99007,3.15,0.31,12.40
2303,5.6,0.250,0.26,3.60,0.037,18.0,115.0,0.99040,3.42,0.50,12.60
6173,6.2,0.200,0.26,9.70,0.030,39.0,102.0,0.99080,3.08,0.56,12.90
1631,7.4,0.250,0.36,2.05,0.050,31.0,100.0,0.99200,3.19,0.44,10.80


=== Describe ===


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol
count,5320.000000,5320.000000,5320.000000,5320.000000,5320.000000,5320.000000,5320.000000,5320.000000,5320.000000,5320.000000,5320.000000
mean,7.215179,0.344130,0.318494,5.048477,0.056690,30.036654,114.109023,0.994535,3.224664,0.533357,10.549241
std,1.319671,0.168248,0.147157,4.500180,0.036863,17.805045,56.774223,0.002966,0.160379,0.149743,1.185933
min,3.800000,0.080000,0.000000,0.600000,0.009000,1.000000,6.000000,0.987110,2.720000,0.220000,8.000000
25%,6.400000,0.230000,0.240000,1.800000,0.038000,16.000000,74.000000,0.992200,3.110000,0.430000,9.500000
50%,7.000000,0.300000,0.310000,2.700000,0.047000,28.000000,116.000000,0.994650,3.210000,0.510000,10.400000
75%,7.700000,0.410000,0.400000,7.500000,0.066000,41.000000,153.250000,0.996770,3.330000,0.600000,11.400000
max,15.900000,1.580000,1.660000,65.800000,0.611000,289.000000,440.000000,1.038980,4.010000,2.000000,14.900000


=== Info ===
<class 'pandas.core.frame.DataFrame'>
Index: 5320 entries, 0 to 6496
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed_acidity         5320 non-null   float64
 1   volatile_acidity      5320 non-null   float64
 2   citric_acid           5320 non-null   float64
 3   residual_sugar        5320 non-null   float64
 4   chlorides             5320 non-null   float64
 5   free_sulfur_dioxide   5320 non-null   float64
 6   total_sulfur_dioxide  5320 non-null   float64
 7   density               5320 non-null   float64
 8   pH                    5320 non-null   float64
 9   sulphates             5320 non-null   float64
 10  alcohol               5320 non-null   float64
dtypes: float64(11)
memory usage: 498.8 KB
None


### 2.3 Variáveis Dependentes (targets)

- **Quality:** Uma nota de 0 a 10 para a qualidade do vinho.
- **Color:** Cor/Tipo do vinho.

In [103]:
display(wine_df[['quality', 'color']].head(7))

,quality,color
0,5,red
1,5,red
2,5,red
3,6,red
4,5,red
5,5,red
6,5,red


In [104]:
# targets -> quality
y_quality_df = wine_uniq_df['quality']
print("=== Head ===")
display(y_quality_df.sample(10))
print("=== Describe ===")
display(y_quality_df.describe())
print("=== Info ===")
print(y_quality_df.info())

=== Head ===


,quality
419,5
2954,5
3024,6
120,5
4865,6
2472,3
4080,6
4358,6
5811,4
3194,6


=== Describe ===


,quality
count,5320.000000
mean,5.795677
std,0.879772
min,3.000000
25%,5.000000
50%,6.000000
75%,6.000000
max,9.000000


=== Info ===
<class 'pandas.core.series.Series'>
Index: 5320 entries, 0 to 6496
Series name: quality
Non-Null Count  Dtype
--------------  -----
5320 non-null   int64
dtypes: int64(1)
memory usage: 83.1 KB
None


In [105]:
# targets -> color
y_color_df = wine_uniq_df['color']
print("=== Head ===")
display(y_color_df.sample(10))
print("=== Describe ===")
display(y_color_df.describe())
print("=== Info ===")
print(y_color_df.info())

=== Head ===


,color
6314,white
6181,white
2155,white
1541,red
5178,white
3772,white
449,red
5497,white
4119,white
937,red


=== Describe ===


,color
count,5320
unique,2
top,white
freq,3961


=== Info ===
<class 'pandas.core.series.Series'>
Index: 5320 entries, 0 to 6496
Series name: color
Non-Null Count  Dtype 
--------------  ----- 
5320 non-null   object
dtypes: object(1)
memory usage: 83.1+ KB
None


## 3. Análise Exploratória de Dados (EDA)

### 3.1 Análise Univariada

Avaliação de cada variável isoladamente (*features* e *targets*).

In [106]:
# Features
# Grid
fig_features = make_subplots(rows=4, cols=3, subplot_titles=X_df.columns)

row = 1
col = 1
for feature in X_df.columns:
    fig_features.add_trace(
        go.Histogram(
            x=X_df[feature],
            name=feature,
            nbinsx=30,
            marker=dict(line=dict(color='black', width=1))
        ),
        row=row,
        col=col
    )
    col += 1
    if col > 3:
        col = 1
        row += 1

fig_features.update_layout(
    title_text="Distribuição das Variáveis Físico-Químicas (Features)",
    showlegend=False,
    # Colab: largura flexível e fixar a altura
    autosize=True,
    height=900
)

# config='responsive' permite que o Plotly preencha a largura da célula no notebook
fig_features.show(config={'responsive': True})

- `pH` e `density` apresentam dados similares a uma distribuição normal.
- `residual_sugar`, `chlorides` e `free_sulfur_dioxide` apresentam assimetria (*skewness*) positiva, com presença de medições extremas (*outliers*) à direita.
- `fixed_acidity`, `volatile_acidity`, `citric_acid` e `sulphates` apresentam assimetria positiva (cauda longa à direita) e presença de *outliers* na cauda superior.
- `alcohol` apresenta assimetria positiva, com pico de concentração em torno de 9,5% - 10% e queda gradual até 14%.

O fato de múltiplas variáveis apresentarem assimetria positiva (*right-skewness*) simultaneamente sugere fortes hipóteses de relacionamento entre as *features* e indica padrões estruturais no processo de geração dos dados

In [107]:
# Targets
## Quality
fig_quality = px.histogram(
    y_quality_df,
    x='quality',
    title='Distribuição da Qualidade (Target de Regressão/Classificação)',
    text_auto=True,
    color_discrete_sequence=['#832232'] # Tom de vinho
)

fig_quality.update_traces(marker_line_color='black', marker_line_width=1)

fig_quality.update_layout(
    autosize=True,
    height=400
)

fig_quality.show(config={'responsive': True})

## Color
fig_color = px.histogram(
    y_color_df,
    x='color',
    title='Distribuição da Cor dos Vinhos (Target de Classificação)',
    text_auto=True,
    color='color',
    color_discrete_map=WINE_COLORS
)

fig_color.update_traces(marker_line_color='black', marker_line_width=1)

fig_color.update_layout(
    autosize=True,
    height=400
)

fig_color.show(config={'responsive': True})

- `quality` apresenta uma distribuição discreta com aparente simetria em torno do valor central 6 (concentrando a maior parte das amostras entre as notas 5 e 7), mas sofre de um nítido **desbalanceamento de classes** nas extremidades.

### 3.2 Análise de Valores Discrepantes (Outliers)

Evidenciar pontos fora da curva.

#### 3.2.1 ph e density

In [108]:
grupo_1 = ['pH', 'density']

for feature in grupo_1:
    fig = px.histogram(
        wine_uniq_df,
        x=feature,
        marginal='violin',
        title=f'Grupo 1 - Variáveis Continuas de Tendência Central: {feature}',
        color_discrete_sequence=['#2E8B57'],
        opacity=0.8
    )

    # Estilização Violin
    fig.update_traces(
        box_visible=True,
        box_fillcolor='white',
        meanline_visible=True,
        meanline_color='darkred',
        line_color='#2E8B57',
        line_width=1.5,
        marker_symbol='circle',
        marker_color='#8B0000',
        selector=dict(type='violin')
    )

    # Estilização Histogram
    fig.update_traces(
        marker_line_color='black',
        marker_line_width=1,
        selector=dict(type='histogram')
    )

    fig.update_layout(
        yaxis2=dict(domain=[0.70, 1.0]),
        yaxis=dict(domain=[0.0, 0.65]),
        bargap=0.1,
        autosize=True,
        height=550
    )

    fig.show(config={'responsive': True})

##### Percepções

- Tanto o `pH` quanto a `density` concentram a imensa maioria das suas observações em um formato simétrico de "sino". A média (linha tracejada vermelha) e a mediana (linha contínua) estão extremamente próximas, confirmando uma forte tendência central.
- No histograma da `density`, percebe-se um topo ligeiramente "achatado/bimodal" reflexo da mistura das duas categorias no dataset (vinhos vermelhos e brancos possuem densidades levemente distintas).
- A densidade do vinho é diretamente afetada pelo teor de açúcar e álcool. Vinhos doces/sobremesa retêm alta densidade, o que explica os pontos extremos sem necessariamente indicar erro de digitação.
- **Para o KNN:** A presença dos outliers extremos em `density` torna o uso de **`StandardScaler`** ou **`RobustScaler`** estritamente obrigatório. Sem o escalonamento, esses valores de densidade distorcerão os cálculos de distância euclidiana.
- **Para a Random Forest:** O algoritmo lidará com ambas as variáveis nativamente sem necessidade de remoção ou transformação prévia.

#### 3.2.2 residual sugar, chlorides e free_sulfur_dioxid

In [109]:
# Assimetria Severa
grupo_2 = ['residual_sugar', 'chlorides', 'free_sulfur_dioxide']

for feature in grupo_2:
  fig = px.histogram(
      wine_uniq_df,
      x=feature,
      marginal='violin',
      title=f'Grupo 2 - Assimetria Severa: {feature}',
      color_discrete_sequence=['#9932CC'],
      opacity=0.8
  )

  # Estilização Violin
  fig.update_traces(
      box_visible=True,
      box_fillcolor='white',
      meanline_visible=True,
      meanline_color='darkred',
      line_color='#9932CC',
      line_width=1.5,
      marker_symbol='circle',
      marker_color='#8B0000',
      selector=dict(type='violin')
  )

  # Estilização Histogram
  fig.update_traces(
      marker_line_color='black',
      marker_line_width=1,
      selector=dict(type='histogram')
  )

  fig.update_layout(
      yaxis2=dict(domain=[0.70, 1.0]),
      yaxis=dict(domain=[0.0, 0.65]),
      bargap=0.1,
      autosize=True,
      height=550
  )

  fig.show(config={'responsive': True})

##### Percepções

- Todas as variáveis deste grupo apresentam concentração massiva de valores no limite inferior, com uma cauda longa se estendendo no sentido positivo.
- Em todos os violinos, a linha tracejada vermelha (Média) está visivelmente deslocada à direita em relação à mediana (centro da caixa branca). Isso demonstra como os valores extremos inflam a média aritmética.
- Picos secundários nos histogramas marcam a diferença fundamental entre tipos de vinho:
  - *Açúcar:* Vinhos secos (pico primário em torno de 1-3 g/L) vs. vinhos doces/suaves (platô em torno de 15-20 g/L).
  - *Cloretos:* Vinhos brancos tendem a ter níveis menores e mais homogêneos, enquanto os vermelhos exibem maior dispersão salina.
- Os pontos isolados no extremo da escala podem representar produtos de nicho.

- **KNN:** Por apresentarem distribuições altamente assimétricas e com grandes amplitudes de valores, o escalonamento simples (`StandardScaler`) pode ter sua média/desvio-padrão distorcidos pelos extremos.
  - O uso de **`RobustScaler`**  ou a aplicação prévia de uma transformação logarítmica ajudará a normalizar estas variáveis para o KNN.
- **Random Forests:** Árvores de decisão não apresentam grandes problemas à assimetria severa e outliers.

#### 3.2.3 fixed_acidity, volatile_acidity e sulphates

In [110]:
# Assimetria Moderada
grupo_3 = ['fixed_acidity', 'volatile_acidity', 'sulphates']

for feature in grupo_3:
  fig = px.histogram(
      wine_uniq_df,
      x=feature,
      marginal='violin',
      title=f'Grupo 3 - Assimetria Moderada: {feature}',
      color_discrete_sequence=['#E06D53'],
      opacity=0.8
  )

  # Estilização Violin
  fig.update_traces(
      box_visible=True,
      box_fillcolor='white',
      meanline_visible=True,
      meanline_color='darkred',
      line_color='#E06D53',
      line_width=1.5,
      marker_symbol='circle',
      marker_color='#8B0000',
      selector=dict(type='violin')
  )

  # Estilização Histogram
  fig.update_traces(
      marker_line_color='black',
      marker_line_width=1,
      selector=dict(type='histogram')
  )

  fig.update_layout(
      yaxis2=dict(domain=[0.70, 1.0]),
      yaxis=dict(domain=[0.0, 0.65]),
      bargap=0.1,
      autosize=True,
      height=550
  )

  fig.show(config={'responsive': True})

##### Percepções

- As três variáveis apresentam o pico principal bem definido na metade esquerda da escala, caindo de forma suave até formar uma cauda longa contínua à direita.
- Em todas as três variáveis, a linha tracejada vermelha (Média) situa-se ligeiramente à direita da mediana e com divergência menor do que observada no Grupo 2, confirmando uma assimetria moderada.
- O histograma da acidez volátil mostra uma pequena elevação secundária em torno de 0.6 - 0.7, que indica novamente a mistura do dataset (vinhos tintos tendem a apresentar maior acidez volátil do que os vinhos brancos).
- Amostras com alta acidez volátil ([acima de 0,6 a 0,8 g/L para brancos e de 0,8 a 1,2 g/L para tintos](https://winefun.com.br/conheca-os-limites-maximos-e-entenda-melhor-a-acidez-volatil-de-seu-vinho/)) são casos raros e indicam maior probabilidade de defeito sensorial (gosto a vinagre/ácido acético).

- **KNN:** Como os dados não têm distorções extremas como a densidade do Grupo 1 ou o açúcar do Grupo 2, o **`StandardScaler`** tradicional funcionará perfeitamente nestas três variáveis para igualar as amplitudes.
- **Para a Random Forest:** A ligeira assimetria não interfere no desempenho do algoritmo baseado em árvores.

#### 3.2.4 citric_acid, total_sulfur_dioxide, alcohol

In [111]:
# Padrões Complexos
grupo_4 = ['citric_acid', 'total_sulfur_dioxide', 'alcohol']

for feature in grupo_4:
  fig = px.histogram(
      wine_uniq_df,
      x=feature,
      marginal='violin',
      title=f'Grupo 4 - Comportamento Complexo / Multimodal: {feature}',
      color_discrete_sequence=['#20B2AA'],
      opacity=0.8
  )

  # Estilização Violin
  fig.update_traces(
      box_visible=True,
      box_fillcolor='white',
      meanline_visible=True,
      meanline_color='darkred',
      line_color='#20B2AA',
      line_width=1.5,
      marker_symbol='circle',
      marker_color='#8B0000',
      selector=dict(type='violin')
  )

  # Estilização Histogram
  fig.update_traces(
      marker_line_color='black',
      marker_line_width=1,
      selector=dict(type='histogram')
  )

  fig.update_layout(
      yaxis2=dict(domain=[0.70, 1.0]),
      yaxis=dict(domain=[0.0, 0.65]),
      bargap=0.1,
      autosize=True,
      height=550
  )

  fig.show(config={'responsive': True})

##### Percepções

- `total_sulfur_dioxide` apresenta dois picos que indicam os dois tipos de vinho: O primeiro agrupamento (< 50 mg/dm³) corresponde aos vinhos vermelhos, enquanto o segundo agrupamento (> 100 mg/dm³) representa os vinhos brancos. [How Much Sulfur is in Wine?](https://winefolly.com/deep-dive/sulfites-in-wine/#:~:text=How%20Much%20Sulfur%20is%20in%20Wine%3F)
  - Assim, os outliers ([+200](https://grape-to-glass.com/index.php/sulphur-levels-wine/)) indicam a presença de vinhos doces, equilibrando o açúcar residual exercendo função antioxidante, antibacteriana e conservatória ([Sulfitos e vinho](https://grape-to-glass.com/index.php/sulphur-levels-wine/))
- `alcohol` é a variável mais distribuida do grupo, com apenas um ponto discrepante extremo próximo a 15.
- **KNN:**: A variável `total_sulfur_dioxide` possui valores que passam de $200$, enquanto `citric_acid` opera na escala de $0$ a $1$. A padronização (`StandardScaler`) é indispensável para que o $SO_2$ não domine completamente o cálculo de distância do KNN.
  - A multimodalidade destas variáveis sugere que a inclusão da variável `color` (ou o split entre vinhos brancos e tintos) ajudará o KNN a separar melhor esses agrupamentos no espaço de atributos.
- **Para a Random Forest:** Árvores de decisão capturam distribuições multimodais e picos discretos com extrema facilidade através de cortes sucessivos (`splits`), tornando a Random Forest um modelo adequado para este grupo de variáveis.

#### 3.2.5 quality e color

In [112]:
# Targets
grupo_5 = ['quality', 'color']

for feature in grupo_5:
  fig = px.histogram(
      wine_uniq_df,
      x=feature,
      marginal='violin',
      title=f'Grupo 5 - Alvos: {feature}',
      color_discrete_sequence=['#775577'],
      opacity=0.8
  )

  # Estilização Violin
  fig.update_traces(
      box_visible=True,
      box_fillcolor='white',
      meanline_visible=True,
      meanline_color='darkred',
      line_color='#775577',
      line_width=1.5,
      marker_symbol='circle',
      marker_color='#8B0000',
      selector=dict(type='violin')
  )

  # Estilização Histogram
  fig.update_traces(
      marker_line_color='black',
      marker_line_width=1,
      selector=dict(type='histogram')
  )

  fig.update_layout(
      yaxis2=dict(domain=[0.70, 1.0]),
      yaxis=dict(domain=[0.0, 0.65]),
      bargap=0.1,
      autosize=True,
      height=550
  )

  fig.show(config={'responsive': True})

##### Percepções

- O dataset é maioritariamente composto por vinhos brancos (`white` ~ 75%) em comparação aos vinhos vermelhos (`red` ~ 25%).
- A imensa maioria das avaliações situa-se nas notas **5 e 6** (que juntas representam mais de 70% do dataset).
* **Escassez de Classes Extremas:** As notas de qualidade extrema (**3, 4, 8 e 9**) contêm pouquíssimas amostras.

- Na divisão de dados (`train_test_split`), será importante estratificar (`stratify=y`) para garantir que as classes raras fiquem proporcionalmente representadas tanto no treino como no teste.
- **KNN e Random Forest:** Avaliar os modelos através de **Macro F1-Score**, **Precision** e **Recall**, e/ou através da matriz de confusão normalizada.

### 3.3 Análise Bivariada e Multivariada

Avaliação da interação entre as variáveis.

In [113]:
# Matriz de correlação apenas com variáveis numéricas
df_eda = wine_uniq_df.copy()

# Encoding Binário (0 = white, 1 = red)
df_eda['color'] = df_eda['color'].map({'white': 0, 'red': 1})

# Matriz de correlação
matriz_corr = df_eda.corr().round(2)

# Heatmap
fig_corr = px.imshow(
    matriz_corr,
    text_auto=True,
    aspect="auto",
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Matriz de Correlação Linear (Incluindo Cor e Qualidade)'
)

fig_corr.update_layout(height=800, autosize=True)
fig_corr.show(config={'responsive': True})

- `free_sulfur_dioxide` e `total_sulfur_dioxide` possuem forte correlação linear, o que já se era esperado por se tratar de componente e soma de dióxido sulfurico.
- Ambos os *targets* `quality` e `color` possuem fraca correlação, indicando que há pouco impacto entre a cor e a qualidade avaliada, o que permite análises mais independentes.
- Para `quality`, aparentemente os principais *features* a se observar seriam `alcohol`, `density` e `volatile_acidity` (níveis elevados levam à semelhança do vinagre).
    - `density` se relaciona inversamente com o próprio percentual de `alcohol` e diretamente com `residual_sugar` (nível de doçura).
- Para `color`, aparentemente os principais *features* a se observar seriam `total_sulfur_dioxide` (níveis elevados levam a um gosto indesejado), `volatile_acidity` (níveis elevados levam à semelhança do vinagre) e `chlorides` (sais dissolvidos).

In [114]:
# Key_features X Quality
key_features_quality = ['alcohol', 'density', 'volatile_acidity', 'chlorides']

fig_matrix_quality = px.scatter_matrix(
    df_eda,
    dimensions=key_features_quality,
    color=df_eda['quality'].astype(str),
    title='Matriz de Dispersão: Perfil Químico por Nota de Qualidade',
    opacity=0.6,
    category_orders={"color": ["3", "4", "5", "6", "7", "8", "9"]},
    color_discrete_sequence=px.colors.sequential.Jet
)

# Oculta a diagonal principal
fig_matrix_quality.update_traces(diagonal_visible=False)
fig_matrix_quality.update_layout(height=800, autosize=True)
fig_matrix_quality.show(config={'responsive': True})

Há um espalhamento bagunçado entre as principais features e seus valores de target, indicando que a previsão da nota do vinho não obedece a regras lineares simples.

In [115]:
# Key_features X Color
key_features_color = ['alcohol', 'density', 'volatile_acidity', 'total_sulfur_dioxide']

fig_matrix = px.scatter_matrix(
    df_eda,
    dimensions=key_features_color,
    color=df_eda['color'].map({0: 'white', 1: 'red'}),
    title='Matriz de Dispersão: Perfil Químico por Cor',
    opacity=0.5,
    color_discrete_map=WINE_COLORS
)

# Oculta a diagonal principal para o gráfico ficar mais limpo
fig_matrix.update_traces(diagonal_visible=False)
fig_matrix.update_layout(height=800, autosize=True)
fig_matrix.show(config={'responsive': True})

As principais *features* quando relacionadas em pares apresentam regiões bem-definidas, mas ainda sim com sobreposição de dados. Isso sugere que a predição do tipo de vinho é matematicamente viável, mas exige uma capacidade de avaliar múltiplas variáveis simultaneamente e segmentar as amostras que caem nas zonas de intersecção química.

#### Percepções Pré-ML

**Relação Físico-Química e Multicolinearidade:** A `density` atua como uma variável dependente do binômio Álcool vs. Açúcar. Vinhos secos e com elevado teor alcoólico apresentam densidades muito baixas, ao passo que vinhos doces (*outliers* de `residual_`sugar $> 15\text{ g/L}$) elevam significativamente a densidade. Como `density`, `residual_sugar` e `alcohol` apresentam forte multicolinearidade e outliers extremos, o `StandardScaler` ou `RobustScaler` é obrigatório para o **KNN** não ser distorcido por estas amplitudes.

**Assinatura Físico-Química por Tipo de Vinho:** A bimodalidade acentuada em `total_sulfur_dioxide` e a assimetria em `free_sulfur_dioxide` são explicadas pelo desequilíbrio de classes da variável color (75% brancos). Vinhos brancos exigem doses substancialmente maiores de sulfitos para evitar a oxidação e preservar a cor clara. A introdução da variável `color` no treino (ou a padronização das feições de enxofre) permitirá aos algoritmos (especialmente o **KNN**) separar com clareza estas duas subpopulações.

**Impacto Organoléptico no Alvo (quality):** A acidez volátil elevada ($> 0.8\text{ g/L}$) é um indicador crítico de degradação acética (sabor a vinagre). A EDA revela que as poucas amostras com acidez volátil alta correlacionam-se com as notas mais baixas da variável alvo quality (notas 3 e 4). Como as classes extremas de qualidade são raras, a divisão dos dados com `stratify=y` no `train_test_split` é indispensável.

## 4. Pré-processamento e Engenharia de Features (Adaptação)

In [116]:
df_original = wine_df.copy()
df_limpo = wine_uniq_df.copy()
df_original['color'] = wine_df['color'].map({'red': 0, 'white': 1})
df_limpo['color'] = wine_uniq_df['color'].map({'red': 0, 'white': 1})

#### Quality x Color

In [117]:
# Dicionário organizando as duas bases de dados
datasets = {
    'Com Duplicadas': wine_df,
    'Sem Duplicadas': wine_uniq_df
}

# 1. Agrupamento Estatístico
lista_estatisticas = [df.groupby('color')['quality'].describe() for df in datasets.values()]
estatisticas_comparacao = pd.concat(lista_estatisticas, keys=datasets.keys())

print("### Resumo Estatístico da Qualidade por Cor\n")
display(
    estatisticas_comparacao
    .round(2)
    .style
    .format("{:.2f}")
    .background_gradient(cmap="Blues", subset=pd.IndexSlice[:, ['mean', 'std']])
    .set_properties(**{'text-align': 'center'})
    .set_caption("Comparação: Com Duplicadas × Sem Duplicadas")
)

print("\n" + "─" * 80 + "\n")

# 2. Tabela de Contingência Proporcional
lista_distribuicao = [
    (pd.crosstab(index=df['color'], columns=df['quality'], normalize='index') * 100).round(2)
    for df in datasets.values()
]
distribuicao_comparacao = pd.concat(lista_distribuicao, keys=datasets.keys()).fillna(0)

print("### Distribuição Percentual das Notas de Qualidade por Cor\n")
display(
    distribuicao_comparacao
    .style
    .format("{:.1f}%")
    .background_gradient(cmap="YlGnBu", axis=1)
    .set_properties(**{'text-align': 'center'})
    .set_caption("Valores em % (por cor)")
)

### Resumo Estatístico da Qualidade por Cor




────────────────────────────────────────────────────────────────────────────────

### Distribuição Percentual das Notas de Qualidade por Cor



- Aparentemente os dataframes com e sem duplicadas não devem constituir uma grande alteração na classificação do modelo. Ainda sim, vale a pena comoparar ambos os casos.
- `quality` será removida para a classificação da cor contar apenas com atributos objetivos.
- Outro ponto importante a se notar é que não há identificadores, seja do vinho, do avaliador, ou mesmo do teste. Isso torna `quality` muito mais subjetiva, além de não ser confiável que as linhas duplicadas sejam apenas coincidências.

#### Análise de Componentes Principais (PCA)

O PCA é uma técnica estatística não supervisionada que reduz o número de dimensões em grandes conjuntos de dados, sendo eficaz para visualizar e explorar conjuntos de dados de alta dimensão ou dados com muitos atributos, pois pode identificar facilmente tendências, padrões ou valores discrepantes.

In [118]:
stroke_map = {
    'red': '#B9979B',   # Borda clara para o tinto
    'white': '#777468'  # Borda escura para o branco
}

# 1. Isolando as features químicas e a variável alvo
X = df_limpo.drop(columns=['color', 'quality'], errors='ignore')
y = df_limpo['color']

# 2. Padronizando os dados
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- PCA 2D ---
pca_2d = PCA(n_components=2)
components_2d = pca_2d.fit_transform(X_scaled)

df_pca_2d = pd.DataFrame(data=components_2d, columns=['PC1', 'PC2'])
df_pca_2d['color'] = y.values
df_pca_2d['color'] = df_pca_2d['color'].map({0: 'white', 1: 'red'})

display(df_pca_2d.head())

# Gráfico 2D interativo com Plotly
fig_2d = px.scatter(
    df_pca_2d, x='PC1', y='PC2', color='color',
    title=f'PCA 2D - Variância Explicada Total: {pca_2d.explained_variance_ratio_.sum()*100:.2f}%',
    color_discrete_map=WINE_COLORS,
    opacity=0.8
)

for trace in fig_2d.data:
    classe = trace.name  # 'red' ou 'white'
    if classe in stroke_map:
        trace.marker.line.width = 1
        trace.marker.line.color = stroke_map[classe]

fig_2d.show()

# --- PCA 3D ---
pca_3d = PCA(n_components=3)
components_3d = pca_3d.fit_transform(X_scaled)

df_pca_3d = pd.DataFrame(data=components_3d, columns=['PC1', 'PC2', 'PC3'])
df_pca_3d['color'] = y.values
df_pca_3d['color'] = df_pca_3d['color'].map({0: 'white', 1: 'red'})

# Gráfico 3D interativo com Plotly
fig_3d = px.scatter_3d(
    df_pca_3d, x='PC1', y='PC2', z='PC3', color='color',
    title=f'PCA 3D - Variância Explicada Total: {pca_3d.explained_variance_ratio_.sum()*100:.2f}%',
    color_discrete_map=WINE_COLORS,
    opacity=0.8
)

for trace in fig_3d.data:
    classe = trace.name  # 'red' ou 'white'
    if classe in stroke_map:
        trace.marker.line.width = 1
        trace.marker.line.color = stroke_map[classe]

fig_3d.show()

,PC1,PC2,color
0,-3.165753,-0.010419,white
1,-3.054962,0.724687,white
2,-3.072476,0.485413,white
3,-1.756467,1.959557,white
4,-2.972872,-0.009583,white


- Observa-se uma que, tanto no gráfico 2D quanto no 3D, os vinhos tintos (red) e brancos (white) formam dois agrupamentos (clusters) bem definidos.
- O PC1 (eixo horizontal no gráfico 2D) é claramente a dimensão responsável por separar a cor: Vinhos brancos concentram-se em valores negativos, enquantoo vinhos tintos concentram-se em valores positivos.
- Para um dataset multidimensional e complexo, conseguir separar quase perfeitamente duas classes utilizando de aproximadamente 50% a 64% da variância total (quão distantes os valores de um conjunto de dados estão em relação à média aritmética) demonstra que as variáveis de maior variância do dataset estão diretamente correlacionadas ao tipo de vinho.

#### Preparação para Classificadores de Cor

In [119]:
df_original = wine_df.copy()
df_limpo = wine_uniq_df.copy()
df_original['color'] = wine_df['color'].map({'red': 0, 'white': 1})
df_limpo['color'] = wine_uniq_df['color'].map({'red': 0, 'white': 1})

In [120]:
def gerar_splits_estratificados(
    df: pd.DataFrame,
    coluna_alvo: str,
    n_splits: int = 11,
    random_state: int = 33
) -> list[tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]]:
    """
    Gera partições de treino e teste estratificadas para validação cruzada,
    garantindo a proporção das classes minoritárias em todos os 'folds'.

    Args:
        df (pd.DataFrame): O DataFrame completo contendo as features e o alvo.
          recebe os dfs com e sem duplicada
        coluna_alvo (str): O nome da coluna que será utilizada como alvo (target).
          espera-se quality ou color
        n_splits (int, optional): O número de partições (folds) para a divisão.
          default=5.
        random_state (int, optional): Semente aleatória para reprodutibilidade.
          default = 33.

    Returns:
        list[tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]]:
            Uma lista contendo `n_splits` tuplas. Cada tupla representa um 'fold'
            e contém os seguintes elementos, nesta exata ordem:
            - X_train (pd.DataFrame): Features de treinamento.
            - X_test (pd.DataFrame): Features de teste.
            - y_train (pd.Series): Alvo de treinamento.
            - y_test (pd.Series): Alvo de teste.
    """

    # Separação das features e do alvo
    X = df.drop(columns=[coluna_alvo])
    y = df[coluna_alvo]

    # Instanciando o separador estratificado
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    splits = []

    # Gerando os índices e construindo as partições
    for train_index, test_index in skf.split(X, y):
        X_train = X.iloc[train_index]
        X_test = X.iloc[test_index]
        y_train = y.iloc[train_index]
        y_test = y.iloc[test_index]

        splits.append((X_train, X_test, y_train, y_test))

    return splits

In [121]:
def calcular_hash_pandas(obj: pd.DataFrame | pd.Series) -> str:
    """
    Gera um hash MD5 único para um DataFrame ou Series do Pandas,
    garantindo a verificação de integridade dos dados.
    """
    # Hash de cada linha do objeto
    hashes_linhas = pd.util.hash_pandas_object(obj, index=True)
    # Combina todos os hashes em uma única string hexadecimal (MD5)
    return hashlib.md5(hashes_linhas.values).hexdigest()

#### Treino e Teste

In [122]:
splits_com_duplicadas = gerar_splits_estratificados(df_original, coluna_alvo='color')
splits_sem_duplicadas = gerar_splits_estratificados(df_limpo, coluna_alvo='color')

# Dicionário prático para o loop de treino
cenarios_cor = {
    'Com Duplicadas': splits_com_duplicadas,
    'Sem Duplicadas': splits_sem_duplicadas
}

In [123]:
for cenario, splits in cenarios_cor.items():
    print(f" === CENÁRIO: {cenario.upper()} === ")

    for i, (X_train, X_test, y_train, y_test) in enumerate(splits):
        # Gerando as assinaturas
        hash_x_train = calcular_hash_pandas(X_train)
        hash_x_test = calcular_hash_pandas(X_test)

        print(f"  [Fold {i+1}]")
        print(f"    - Treino: {X_train.shape[0]:>4} amostras | Hash X_train: {hash_x_train}")
        print(f"    - Teste : {X_test.shape[0]:>4} amostras | Hash X_test : {hash_x_test}")

 === CENÁRIO: COM DUPLICADAS === 
  [Fold 1]
    - Treino: 5906 amostras | Hash X_train: e52d63a793e6c06add09457d16dd470e
    - Teste :  591 amostras | Hash X_test : 3f70382eea7c1bdecbb4877ca87d9980
  [Fold 2]
    - Treino: 5906 amostras | Hash X_train: 5c32911d2041a022f4a38361cc3fefd0
    - Teste :  591 amostras | Hash X_test : 970f12570bc76eb28101544005a19516
  [Fold 3]
    - Treino: 5906 amostras | Hash X_train: c38e93c0d192733dcc552c8db128f7db
    - Teste :  591 amostras | Hash X_test : 06095a5eac787d3bf2d26cae922b570a
  [Fold 4]
    - Treino: 5906 amostras | Hash X_train: 6361e6e6674cb85386ede9b8b7195a8a
    - Teste :  591 amostras | Hash X_test : 8e1d877c83aef9acea3a70f762a71685
  [Fold 5]
    - Treino: 5906 amostras | Hash X_train: c6aa50f2b1670e8e7b1e21ed53ab8104
    - Teste :  591 amostras | Hash X_test : a89c954c3e08f370f612b76dc6bda88a
  [Fold 6]
    - Treino: 5906 amostras | Hash X_train: ea5faf0f0b5faabe3abd51bd1150f402
    - Teste :  591 amostras | Hash X_test : 24b11898e

In [124]:
# Pipeline 1: KNN (Precisa imperativamente do Scaler)
pipeline_knn = Pipeline([
    ('scaler', RobustScaler()),
    ('modelo', KNeighborsClassifier())
])

# Pipeline 2: Random Forest
pipeline_rf = Pipeline([
    ('modelo', RandomForestClassifier(class_weight='balanced', random_state=42))
])

modelos = {
    'KNN': pipeline_knn,
    'Random Forest': pipeline_rf
}

In [125]:
resultados = []

for nome_cenario, splits in cenarios_cor.items():
    print(f" === CENÁRIO: {nome_cenario} === ")

    for nome_modelo, model in modelos.items():
        acuracias_folds = []
        matrizes_confusao = []

        # Iterando sobre cada fold retornado pela sua função
        for i, (X_train, X_test, y_train, y_test) in enumerate(splits, start=1):

            # Treinamento do modelo no fold atual
            model.fit(X_train, y_train)

            # Predição
            y_pred = model.predict(X_test)

            # Avaliação do fold
            acc = accuracy_score(y_test, y_pred)
            cm = confusion_matrix(y_test, y_pred)

            acuracias_folds.append(acc)
            matrizes_confusao.append(cm)

        # Consolidação dos resultados do K-Fold
        media_acc = np.mean(acuracias_folds)
        std_acc = np.std(acuracias_folds)

        # Matriz de confusão acumulada (soma das confusões em todos os folds)
        matriz_acumulada = np.sum(matrizes_confusao, axis=0)

        print(f"\n---> Modelo: {nome_modelo}")
        print(f"Acurácias por fold: {[round(a, 4) for a in acuracias_folds]}")
        print(f"Acurácia Média: {media_acc:.4f} (± {std_acc:.4f})")
        print("Matriz de Confusão Acumulada:")
        print(matriz_acumulada)

        resultados.append({
            'Cenário': nome_cenario,
            'Modelo': nome_modelo,
            'Acurácia Média': media_acc,
            'Desvio Padrão': std_acc
        })

# Resumo em DataFrame
df_resumo = pd.DataFrame(resultados)
print("\n--- RESUMO COMPARATIVO ---")
print(df_resumo)

 === CENÁRIO: Com Duplicadas === 

---> Modelo: KNN
Acurácias por fold: [0.9966, 0.9932, 0.9966, 0.9949, 0.9882, 0.9915, 0.9915, 0.9881, 0.9864, 0.9932, 0.9915]
Acurácia Média: 0.9920 (± 0.0032)
Matriz de Confusão Acumulada:
[[1570   29]
 [  23 4875]]

---> Modelo: Random Forest
Acurácias por fold: [0.9966, 0.9949, 0.9966, 0.9983, 0.9949, 0.9966, 0.9949, 0.9881, 0.9932, 0.9949, 0.9949]
Acurácia Média: 0.9949 (± 0.0025)
Matriz de Confusão Acumulada:
[[1574   25]
 [   8 4890]]
 === CENÁRIO: Sem Duplicadas === 

---> Modelo: KNN
Acurácias por fold: [0.9917, 0.9938, 0.9938, 0.9897, 0.9917, 0.9897, 0.9959, 0.9814, 0.9917, 0.9959, 0.9896]
Acurácia Média: 0.9914 (± 0.0038)
Matriz de Confusão Acumulada:
[[1335   24]
 [  22 3939]]

---> Modelo: Random Forest
Acurácias por fold: [0.9959, 0.9959, 0.9959, 0.9897, 0.9897, 0.9979, 0.9938, 0.9896, 0.9979, 0.9959, 0.9938]
Acurácia Média: 0.9942 (± 0.0030)
Matriz de Confusão Acumulada:
[[1335   24]
 [   7 3954]]

--- RESUMO COMPARATIVO ---
          Ce

- No cenário **Com Duplicadas**, a acurácia do Random Forest foi levemente inflada para 0.9949 (99,49%), enquanto no cenário **Sem Duplicadas**, a acurácia foi de 0.9942 (99,42%).
    -  A queda na acurácia foi mínima (~0,07%), indicando que os modelos não estavam apenas "decorando" dados duplicados.
- O **Random Forest** superou o **KNN** em ambos os cenários (maior acurácia média e menor desvio padrão).
    - No cenário **Sem Duplicadas**, ele errou apenas 31 amostras em um total de 5.320 vinhos (7 vinhos de uma cor classificados como a outra e 24 vice-versa).
- Conclui-se que os modelos obtiveram excelente resultado, cumprindo a separação das duas categorias com elevada precisão. Esta elevada performance é diretamente explicada pela análise de componentes principais (PCA):
    - Os gráficos de PCA (2D e 3D) revelam uma separação espacial e linear quase perfeita entre os vinhos tintos e brancos, com uma concentração distinta ao longo do primeiro componente principal (PC1).
    - O facto de apenas 2 a 3 dimensões reterem entre 49.67% e 64.10% da variância total e já garantirem dois clusters praticamente disjuntos confirma que as propriedades químicas fundamentais do vinho possuem uma forte assinatura biológica/química associada à sua cor.
    - Os escassos erros cometidos pelo modelo correspondem, visualmente, aos outliers químicos observados nas periferias dos *clusters* do PCA, referentes a vinhos com perfis analíticos atípicos para a sua categoria.

In [126]:
X_final = df_original.drop(columns=['color', 'quality'], errors='ignore')
y_final = df_original['color']

# Salvando o Pipeline: garante padronização em dados futuros
modelo_campeao = pipeline_rf

modelo_campeao.fit(X_final, y_final)

nome_arquivo = 'gcmodel_wine_color_rf.joblib'
joblib.dump(modelo_campeao, nome_arquivo)

print(f"Modelo final (Random Forest Pipeline) salvo com sucesso como '{nome_arquivo}'!")

Modelo final (Random Forest Pipeline) salvo com sucesso como 'gcmodel_wine_color_rf.joblib'!


## 5. Conclusão e Considerações Finais

### 5.1 Resposta às Hipóteses e Análise Exploratória (EDA)

A análise exploratória e os testes estatísticos/químicos demonstraram que os vinhos tintos e brancos possuem perfis analíticos significativamente distintos, principalmente em atributos como **total_sulfur_dioxide** e **chlorides**.

### 5.2 Validação da Redução de Dimensionalidade (PCA)

A aplicação do PCA (2D e 3D) confirmou visualmente a hipótese da EDA:
- O primeiro componente principal (**PC1**) foi capaz de separar espacialmente os dois tipos de vinho em clusters praticamente disjuntos.
- A retenção de **49.67% (2D)** a **64.10% (3D)** da variância total do dataset foi suficiente para demonstrar a alta separabilidade química das duas classes.

### 5.3. Comparação de Desempenho e Escolha do Modelo

- Ambos os modelos (**KNN** e **Random Forest**) atingiram o requisito do projeto (acurácia > 85%), ultrapassando **99% de acurácia média**.
- O **Random Forest** apresentou desempenho superior no cenário **Com Duplicadas**:
  - **Acurácia Média (11-Fold CV):** 99,49% (± 0,25%)
  - **Matriz de Confusão:** Erro em apenas 33 amostras de um total de 6464 vinhos.

### 5.4. Modelo em Produção

O pipeline completo (pré-processamento + classificador Random Forest) foi treinado em 100% dos dados originas, considerando que as duplicadas possam não serem um erro, e persistido no arquivo `gcmodel_wine_color_rf.joblib` para utilização posterior em novos dados químicos.